# 1. 실전 예상문제
제공된 학습용 데이터(elec_train.csv)는 전국 건물의 기상 정보(기온, 강수량, 풍속, 습도) 및 전력 소비량을 기록한 자료이다. 해당 데이터를 기반으로 전력 소비량을 예측하는 회귀 모델을 개발하고, 가장 우수한 모델을 평가 데이터(elec_test.csv)에 적용하여 전력 소비량을 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함

[제공 데이터]
1. elec_train.csv : 학습용 데이터 (5,100건)
2. elec_test.csv : 평가용 데이터 (630건, 전력 소비량 컬럼 미제공)

[데이터 컬럼]
1. 건물코드 : 건물 고유코드(100개 코드 범주)
2. 기온 : 일 평균 기온
3. 강수량 : 일 누적 강수량
4. 풍속 : 평균 풍속
5. 습도 : 평균 습도
6. 전력소비량 : 평균 소비 전력(kWh)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred
3. 예측 결과 개수 : 630




In [ ]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.


# 1. 풀이(#1)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['건물코드'].value_counts())
#print(test['건물코드'].value_counts())

# X, Y 데이터 셋 분리
X_train = train.drop(['전력소비량'], axis=1)
y = train['전력소비량']

#print(X_train.shape, y.shape, test.shape)

# 결측치 처리
X_train['강수량'] = X_train['강수량'].fillna(0)
X_train['풍속'] = X_train['풍속'].fillna(X_train['풍속'].mean())
X_train['습도'] = X_train['습도'].fillna(X_train['습도'].mode()[0])

#print(X_train.isnull().sum())

# 수치형 변수 스케일링 - MinMaxScaling
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_columns = X_train.select_dtypes(exclude='object').columns
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
test[num_columns] = scaler.transform(test[num_columns])
#print(X_train.head())

# 범주형 변수 인코딩 - LabelEncoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X_train['건물코드'] = encoder.fit_transform(X_train['건물코드'])
test['건물코드'] = encoder.transform(test['건물코드'])
#print(X_train['건물코드'])

# 학습, 검증 데이터 셋 분할
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

865.8009339119728 0.897413817074046


# 1. 풀이(#2)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/elec_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['건물코드'].value_counts())
#print(test['건물코드'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['전력소비량'], axis=1)
y = train['전력소비량']

X_full = pd.concat([X, test], axis=0)
#print(X_full.shape)

# 결측치 처리
X_full['강수량'] = X_full['강수량'].fillna(0)
X_full['풍속'] = X_full['풍속'].fillna(X_full['풍속'].mean())
X_full['습도'] = X_full['습도'].fillna(X_full['습도'].mode()[0])

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - OneHotEncoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

880.8950192363885 0.87024855682422


# 2. 실전 예상문제
제공된 학습용 데이터(house_price_train.csv)는 주택의 다양한 특성 정보와 실제 판매 가격(SlaePrice)을 기록한 자료이다. 해당 데이터를 기반으로 주택 가격을 예측하는 회귀 모델을 개발하고, 가장 우수한 모델을 평가 데이터(elec_test.csv)에 적용하여 주택 가격을 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함

[제공 데이터]
1. house_price_train.csv : 학습용 데이터 (1,500건)
2. house_price_test.csv : 평가용 데이터 (1,195건, SalePrice 컬럼 미제공)

[데이터 컬럼]
1. Id : 주택 식별자 (고유 번호)
2. OverallQual : 전반적 마감 품질 등급
3. GarageCars : 차고 수용 차량 수
4. GarageArea : 차고 면적 (제곱 피트)
5. YearBuilt : 건축 연도
6. YearRemoveAdd : 리모델링 연도
7. KitchenQual : 주방 품질 등급
8. .... 기타 여러가지 특성
9. SalePrice : 주택 판매 가격

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred
3. 예측 결과 개수 : 1,195




In [ ]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.


# 2. 풀이(#1)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Neighborhood'].value_counts())
#print(test['Neighborhood'].value_counts())
#print(train['KitchenQual'].value_counts())
#print(test['KitchenQual'].value_counts())

# X,Y 데이터 셋 분리
X_train = train.drop(['Id', 'SalePrice'], axis=1)
y = train['SalePrice']
X_test = test.drop(['Id'], axis=1)

#print(X_train.shape, y.shape, X_test.shape)

# 결측치 처리
X_train['LotFrontage'] = X_train['LotFrontage'].fillna(X_train['LotFrontage'].mean())
#print(X_train.isnull().sum())

# 수치형 변수 스케일링 - StandardScaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_columns = X_train.select_dtypes(exclude='object').columns
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_test[num_columns] = scaler.transform(X_test[num_columns])
#print(X_train.head())

# 범주형 변수 인코딩 - LabelEncoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X_train['Neighborhood'] = encoder.fit_transform(X_train['Neighborhood'])
X_test['Neighborhood'] = encoder.transform(X_test['Neighborhood'])
X_train['KitchenQual'] = encoder.fit_transform(X_train['KitchenQual'])
X_test['KitchenQual'] = encoder.transform(X_test['KitchenQual'])
#print(X_train['Neighborhood']

# 학습, 검증 데이터 셋 분할
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# XGBoost 활용 모델 학습
from xgboost import XGBRegressor
model = XGBRegressor()
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

33829.38218050148 0.8401744033315469


# 2. 풀이(#2)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/house_price_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Neighborhood'].value_counts())
#print(test['Neighborhood'].value_counts())
#print(train['KitchenQual'].value_counts())
#print(test['KitchenQual'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['SalePrice'], axis=1)
y = train['SalePrice']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['Id'], axis=1)
#print(X_full.shape)

# 결측치 처리
X_full['LotFrontage'] = X_full['LotFrontage'].fillna(X_full['LotFrontage'].mean())

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(X_train, y_train)

# RMSE, R2 Score 활용 평가
from sklearn.metrics import root_mean_squared_error, r2_score
y_pred = model.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(rmse, r2)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

35619.53337347391 0.7799314679637132
